## Loading imports and data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from FPCA import FPCA, Phimaker, omegacomputer, penaltymatrixmaker
import pandas as pd
import matplotlib.dates as mdates
from tqdm import tqdm
from pathlib import Path

In [ ]:
def partmissing(sensorseries):
    mask = ~np.isnan(sensorseries)
    if not mask.any(): return 100
    start_idx = np.argmax(mask)
    end_idx = len(sensorseries) - np.argmax(mask[::-1]) - 1
    series = sensorseries[start_idx:end_idx+1]
    return np.isnan(series).sum()/len(series) * 100

def partoutlier(sensorseries):
    mask = ~np.isnan(sensorseries)
    if not mask.any(): return None
    start_idx = np.argmax(mask)
    end_idx = len(sensorseries) - np.argmax(mask[::-1]) - 1
    series = sensorseries[start_idx:end_idx+1]

    nonmissing = np.sum(~np.isnan(series))
    outlier_count = np.sum(series > 500)
    return outlier_count / nonmissing * 100

def get_start_percentage(sensorseries):
    mask = ~np.isnan(sensorseries)
    start_idx = np.argmax(mask)
    return (start_idx / len(sensorseries)) * 100

def longest_high_run(sensorseries, threshold=500):
    mask = ~np.isnan(sensorseries)
    if not mask.any(): return None
    start_idx = np.argmax(mask)
    end_idx = len(sensorseries) - np.argmax(mask[::-1]) - 1
    series = sensorseries[start_idx:end_idx+1]

    high = series > threshold

    max_run = 0
    current = 0
    for x in high:
        if x:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run

def dataloader():
    raw_pollution = pd.read_csv("../../../purpleair/purpleair_download/europe_pm25_wide_WHOLE.csv", index_col=0)
    sensors = pd.read_csv("../../../purpleair/purpleair_download/europe_selected_sensors.csv", index_col=0)
    
    # Step 1: throw away completely useless sensors
    # Keep only viable sensors
    missing_rates = raw_pollution.apply(partmissing, axis=1)
    sensors['missing_rate'] = sensors.index.map(missing_rates)
    sensors = sensors[sensors['missing_rate'] < 50]
    raw_pollution = raw_pollution.loc[sensors.index]

    # Removing extreme outlier cases
    outlier_rates = raw_pollution.apply(partoutlier, axis=1)
    sensors['outlier_rate'] = sensors.index.map(outlier_rates)
    sensors = sensors[sensors['outlier_rate'] < 25]
    raw_pollution = raw_pollution.loc[sensors.index]

    # Removing consistent outlier sensors
    runs = raw_pollution.apply(longest_high_run, axis=1)
    sensors["max_high_run"] = sensors.index.map(runs)
    sensors = sensors[sensors.max_high_run < 14]
    raw_pollution = raw_pollution.loc[sensors.index]

    # Step 2: retrieving data that is viable for evaluating, but not usefull for training
    start_percentages = raw_pollution.apply(get_start_percentage, axis=1)
    sensors['start_at_percentage'] = raw_pollution.index.map(start_percentages)
    sensors_extra = sensors[sensors['start_at_percentage'] >= 50]
    sensors = sensors[sensors['start_at_percentage'] < 50]
    pollution_extra = raw_pollution.loc[sensors_extra.index]
    pollution = raw_pollution.loc[sensors.index]

    # Step 3: collecting all data
    dates = pd.to_datetime(raw_pollution.columns)
    longitude_extra = sensors_extra["longitude"].values
    latitude_extra = sensors_extra["latitude"].values
    pollution_extra = pollution_extra.values.T
    longitude = sensors["longitude"].values
    latitude = sensors['latitude'].values
    pollution = pollution.values.T
    pollution[pollution == 0] = np.nan

    return dates, longitude, latitude, pollution, longitude_extra, latitude_extra, pollution_extra

## Model specification

In [ ]:
def pooledgammaest(Y_list, Phi, R, l=0.1):
    Phistar = []
    ystar = []
    for y in Y_list:
        mask = ~np.isnan(y)
        y_obs = y[mask]
        ystar.append(y_obs)
        Phi_obs = Phi[mask]
        Phistar.append(Phi_obs)

    ystar = np.concatenate(ystar)
    Phistar = np.concatenate(Phistar)
    if Phi.shape[1] < 49:
        return np.linalg.lstsq(Phistar, ystar, rcond=None)[0]
    else:
        return np.linalg.solve(Phistar.T @ Phistar + l * R, Phistar.T @ ystar)

def Kalmanfilter(Y_list, Phi, T, c, vareta, vareps, R):
    days, _ = Y_list.shape
    K = Phi.shape[1]

    g_pred = np.zeros((days, K))
    P_pred = np.zeros((days, K, K))
    g_filt = np.zeros((days, K))
    P_filt = np.zeros((days, K, K))

    g_pred[0] = pooledgammaest(Y_list, Phi, R)
    P_pred[0] = np.eye(K)

    for t in range(days):
        y = Y_list[t]
        valid_mask = ~np.isnan(y)
        y_obs = y[valid_mask]
        Phi_obs = Phi[valid_mask]

        F_t  = Phi_obs @ P_pred[t] @ Phi_obs.T + vareps * np.eye(len(Phi_obs))
        v_t = y_obs - Phi_obs @ g_pred[t]
        K_t = P_pred[t] @ Phi_obs.T @ np.linalg.inv(F_t)
        g_filt[t] = g_pred[t] + K_t @ v_t
        P_filt[t] = P_pred[t] - K_t @ Phi_obs @ P_pred[t]

        if t < days - 1:
            g_pred[t+1] = T @ g_filt[t] + c
            P_pred[t+1] = T @ P_filt[t] @ T.T + vareta * np.eye(K)

    return g_pred, P_pred, g_filt, P_filt

def Kalmansmoother(Y_list, Phi, g_pred, P_pred, vareps, vareta, B):
    days, _ = Y_list.shape
    K = Phi.shape[1]

    g_smooth = np.zeros_like(g_pred)
    P_smooth = np.zeros_like(P_pred)
    P_lag = np.zeros((days, K, K))


    r = np.zeros((days, K))
    N = np.zeros((days, K, K))

    r[-1] = 0
    N[-1] = 0

    for t in reversed(range(days)):
        y = Y_list[t]
        valid_mask = ~np.isnan(y)
        y_obs = y[valid_mask]
        Phi_obs = Phi[valid_mask]

        F_t  = Phi_obs @ P_pred[t] @ Phi_obs.T + vareps * np.eye(len(Phi_obs))
        K_t = B @ P_pred[t] @ Phi_obs.T @ np.linalg.inv(F_t)
        L_t = B - K_t @ Phi_obs
        v_t = y_obs - Phi_obs @ g_pred[t]

        if t > 0:
            r[t-1] = Phi_obs.T @ np.linalg.inv(F_t) @ v_t + L_t.T @ r[t]
            g_smooth[t] = g_pred[t] + P_pred[t] @ r[t-1]

            N[t-1] = Phi_obs.T @ np.linalg.inv(F_t) @ Phi_obs + L_t.T @ N[t] @ L_t
            P_smooth[t] = P_pred[t] - P_pred[t] @ N[t-1] @ P_pred[t]

        if t <  days - 1:
            P_lag[t] = P_pred[t] @ L_t.T @ (np.eye(K) - N[t] @ P_pred[t+1])
    return g_smooth, P_smooth, P_lag


def emUpdate(g_smooth, P_smooth, P_lag, Y_list, Phi):
    days, K = g_smooth.shape
    N, _ = Phi.shape

    ##### estimation of B and omega #######
    cross_moments = np.zeros((K, K + 1))
    auto_moments = np.zeros((K + 1, K + 1))

    for t in range(days-1):
        g_t = g_smooth[t]
        g_tp1 = g_smooth[t + 1]
        P_t = P_smooth[t]
        P_lag_t = P_lag[t]

        # E[z_t z_t^T]
        E_zt_ztT = np.block([
            [np.ones((1, 1)), g_t[None, :]],
            [g_t[:, None], P_t + np.outer(g_t, g_t)]
        ])

        # E[gamma_{t+1} z_t^T]
        E_gammatp1_ztT = np.hstack([
            g_tp1[:, None],
            P_lag_t.T + np.outer(g_tp1, g_t)
        ])

        cross_moments += E_gammatp1_ztT
        auto_moments += E_zt_ztT

    Theta_hat = cross_moments @ np.linalg.inv(auto_moments)  # (K, K+1)
    B_hat = Theta_hat[:, 1:]
    omega_hat = Theta_hat[:, 0]

    ##### estimation of eta #######
    sum_eta = 0
    for t in range(days-1):
        messy_var = P_smooth[t+1] + B_hat @ P_smooth[t] @ B_hat.T - P_lag[t].T @ B_hat.T - B_hat @ P_lag[t]
        etares = g_smooth[t+1] - (omega_hat + B_hat @ g_smooth[t])
        sum_eta += etares.T @ etares + np.trace(messy_var)

    vareta_hat = sum_eta / (K * (days -1 ))


    ##### estimation of epsilon #######
    sum_eps = 0
    for t in range(days):
        y = Y_list[t]
        valid_mask = ~np.isnan(y)
        y_obs = y[valid_mask]
        Phi_obs = Phi[valid_mask]

        epsres = y_obs - Phi_obs @ g_smooth[t]
        epsvar = Phi_obs @ P_smooth[t] @ Phi_obs.T
        sum_eps += epsres.T @ epsres + np.trace(epsvar)

    vareps_hat = sum_eps / (N * days)

    return B_hat, omega_hat, vareta_hat, vareps_hat

def em(Y_list, Phi, R, n_iter=10):
    K = Phi.shape[1]
    B_hat = np.eye(K) * 0.05
    omega_hat = np.zeros(K)
    vareta_hat = 1
    vareps_hat = 1

    for i in range(n_iter):
        g_pred, P_pred, _, _ = Kalmanfilter(Y_list, Phi, B_hat, omega_hat, vareta_hat, vareps_hat, R)
        g_smooth, P_smooth, P_lag = Kalmansmoother(Y_list, Phi, g_pred, P_pred, vareps_hat, vareta_hat, B_hat)
        B_hat, omega_hat, vareta_hat, vareps_hat = emUpdate(g_smooth, P_smooth, P_lag, Y_list, Phi)

    return B_hat, omega_hat, vareta_hat, vareps_hat

def onestepforecast(Y_list_train, Y_list_test, Phi, Phi_surface, B_hat, omega_hat, vareta_hat, vareps_hat, R):
    K = Phi.shape[1]
    Gamma_forecast = np.zeros((len(Y_list_test), K))
    history = Y_list_train
    for i in tqdm(range(len(Y_list_test))) :
      g_pred, P_pred, g_filt, P_filt = Kalmanfilter(history[-100:], Phi, B_hat, omega_hat, vareta_hat, vareps_hat, R)
      Gamma_forecast[i] = B_hat @ g_filt[-1] + omega_hat
      history = np.vstack((history, Y_list_test[i:i+1]))

    return Gamma_forecast @ Phi.T, Gamma_forecast @ Phi_surface.T


def insampleforecaster(Y_list_train, Phi, Phi_surface, B_hat, omega_hat, vareta_hat, vareps_hat, R):
    days, N = Y_list_train.shape
    K = Phi.shape[1]

    Gamma_insample = np.zeros((days, K))

    for t in tqdm(range(days)):
        history = Y_list_train[max(0, t-99):t+1]

        g_pred, P_pred, g_filt, P_filt = Kalmanfilter(
            history, Phi, B_hat, omega_hat, vareta_hat, vareps_hat, R
        )

        Gamma_insample[t] = B_hat @ g_filt[-1] + omega_hat

    return Gamma_insample @ Phi.T, Gamma_insample @ Phi_surface.T

## Performing FPCA

In [ ]:
def rescaler(x, eps = 0.01): return eps + (1 - 2*eps) * (x - np.min(x)) / (np.max(x) - np.min(x))

In [ ]:
dates, longitude, latitude, pollution, _, _, _ = dataloader()
eval_indices = np.arange(len(longitude)-75, len(longitude))
train_indices = np.arange(len(longitude)-75)
s1 = rescaler(longitude[train_indices])
s2 = rescaler(latitude[train_indices])
s1_eval = rescaler(longitude[eval_indices])
s2_eval = rescaler(latitude[eval_indices])
Phi = Phimaker(s1, s2)
Phi_surface = Phimaker(s1_eval, s2_eval)

Omega0, Omega1, Omega2 = omegacomputer()
R = penaltymatrixmaker(Omega0, Omega1, Omega2)

Y_list_surface = np.log(pollution[:, eval_indices])
Y_list = np.log(pollution[:, train_indices])
Y_list_train = Y_list[:-365]
Y_list_test = Y_list[-365:]


T, N = Y_list.shape
r = 7
_, Phi_eig, Phi_surface_eig, Level, Level_eval = FPCA(Y_list_train, Phi, Phi_surface, r)
Level_list = np.tile(Level, (T,1))
Level_list_eval = np.tile(Level_eval, (T,1))
Y_list = Y_list - Level_list
Y_list_surface = Y_list_surface - Level_list_eval
Y_list_train = Y_list[:-365]
Y_list_test = Y_list[-365:]
Y_list_surface_test = Y_list_surface[-365:]

## Model fit: Dimensionality reduction

In [ ]:
B_hat, omega_hat, vareta_hat, vareps_hat = em(Y_list_train, Phi_eig, R, n_iter=10)
Mutilde_list_test, Mutilde_list_surface_test = onestepforecast(Y_list_train, Y_list_test, Phi_eig, Phi_surface_eig, B_hat, omega_hat, vareta_hat, vareps_hat, R)
Mutilde_list_train, Mutilde_list_surface_train = insampleforecaster(Y_list_train, Phi_eig, Phi_surface_eig, B_hat, omega_hat, vareta_hat, vareps_hat, R)


Mutilde_list = np.concatenate((Mutilde_list_train, Mutilde_list_test)) + Level_list
Mutilde_list_surface = np.concatenate((Mutilde_list_surface_train, Mutilde_list_surface_test)) + Level_list_eval
Mutilde_list_complete_fpca = np.exp(np.hstack((Mutilde_list, Mutilde_list_surface)))

errors = np.abs(
    np.exp(Mutilde_list_test + Level_list[-365:])
    - np.exp(Y_list_test + Level_list[-365:])
)
MAE = np.nanmean(errors)

errors_surface = np.abs(
    np.exp(Mutilde_list_surface_test + Level_list_eval[-365:])
    - np.exp(Y_list_surface_test + Level_list_eval[-365:])
)
MAE_surface = np.nanmean(errors_surface)


print(f"Out of sample MAE of used sensors: {MAE}")
print(f"Out of sample MAE of unused sensors: {MAE_surface}")

plot_dict = {
    21 : "Milan, Italy (estimation sensor)", 
    62 : "Aegina, Greece (estimation sensor)", 
    164 : "Ljubljana, Slovenia (evaluation sensor)", 
    181 : "Hastings, United Kingdom (evaluation sensor)"}


for idx in plot_dict.keys():
    plt.figure(figsize=(10, 5))
    plt.plot(dates, pollution[:, idx], label=r"$y$", color='#7f7f7f')
    plt.plot(dates, Mutilde_list_complete_fpca[:, idx], label=r"$\tilde{\mu}_{\text{state-space}}$", color='blue', alpha=0.6)
    plt.axvline(x=dates[-365], color='black', linestyle=':', linewidth=2, label="Train/Test Split")
    plt.xlim(dates[0], dates[-1])
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=10))
    plt.legend(fontsize=17)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.3)
    plt.title(f"{plot_dict[idx]}", fontsize=22)
    plt.show()

## Model fit: No dimensionality reduction

In [ ]:
B_hat, omega_hat, vareta_hat, vareps_hat = em(Y_list_train, Phi, R, n_iter=10)
Mutilde_list_test, Mutilde_list_surface_test = onestepforecast(Y_list_train, Y_list_test, Phi, Phi_surface, B_hat, omega_hat, vareta_hat, vareps_hat, R)
Mutilde_list_train, Mutilde_list_surface_train = insampleforecaster(Y_list_train, Phi, Phi_surface, B_hat, omega_hat, vareta_hat, vareps_hat, R)

Mutilde_list = np.concatenate((Mutilde_list_train, Mutilde_list_test)) + Level_list
Mutilde_list_surface = np.concatenate((Mutilde_list_surface_train, Mutilde_list_surface_test)) + Level_list_eval
Mutilde_list_complete_bspline = np.exp(np.hstack((Mutilde_list, Mutilde_list_surface)))

errors = np.abs(
    np.exp(Mutilde_list_test + Level_list[-365:])
    - np.exp(Y_list_test + Level_list[-365:])
)
MAE = np.nanmean(errors)


errors_surface = np.abs(
    np.exp(Mutilde_list_surface_test + Level_list_eval[-365:])
    - np.exp(Y_list_surface_test + Level_list_eval[-365:])
)
MAE_surface = np.nanmean(errors_surface)

print(f"Out of sample MAE of used sensors: {MAE}")
print(f"Out of sample MAE of unused sensors: {MAE_surface}")

plot_dict = {
    21 : "Milan, Italy (estimation sensor)", 
    62 : "Aegina, Greece (estimation sensor)", 
    164 : "Ljubljana, Slovenia (evaluation sensor)", 
    181 : "Hastings, United Kingdom (evaluation sensor)"}


for idx in plot_dict.keys():
    plt.figure(figsize=(10, 5))
    plt.plot(dates, pollution[:, idx], label=r"$y$", color='#7f7f7f')
    plt.plot(dates, Mutilde_list_complete_bspline[:, idx], label=r"$\tilde{\mu}_{\text{state-space, B-spline}}$", color='blue', alpha=0.35)
    plt.plot(dates, Mutilde_list_complete_fpca[:, idx], label=r"$\tilde{\mu}_{\text{state-space, FPCA}}$", color='red', alpha=0.35)
    plt.axvline(x=dates[-365], color='black', linestyle=':', linewidth=2, label="Train/Test Split")
    plt.xlim(dates[0], dates[-1])
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=10))
    plt.legend(fontsize=17)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.3)
    plt.title(f"{plot_dict[idx]}", fontsize=22)
    plt.show()